# Segment 3: Dataset Exemplars — Finding Top-Activating Images for InceptionV1 Channels

## Overview

This notebook implements **Dataset Exemplars** — a core technique in neural network mechanistic interpretability. The goal is to discover **which real images from ImageNet most strongly activate each channel (neuron/feature)** in a given layer of the InceptionV1 (GoogLeNet) model.

### Why Dataset Exemplars?

In Segment 2 (Activation Maximization), we generated *synthetic* images that maximize a neuron's activation. Here, we take the complementary approach: instead of synthesizing images, we **search through the entire ImageNet training set** (~1.28 million images) to find the **real images** that produce the highest activation for each channel. This gives us a grounded understanding of what each feature detector responds to "in the wild."

### What Does This Notebook Do?

The notebook performs two main passes over the ImageNet training dataset:

1. **Pass 1 — Top-K Scan:** Streams through the entire ImageNet training set, computes activations at a target layer (e.g., `mixed4e`), and maintains a **min-heap** per channel to track the top-K highest-activating image keys. This is done efficiently using a single forward pass per batch and heap-based selection.

2. **Pass 2 — Image Retrieval & Saving:** Streams through the dataset a second time, looking for the specific images identified in Pass 1. For each top-K image, it saves:
   - The **full image** (denormalized back to RGB)
   - A **crop** centered on the spatial location of maximum activation for that channel

### Key Concepts

| Concept | Description |
|---------|-------------|
| **Forward Hook** | A PyTorch mechanism to intercept intermediate activations from any layer during a forward pass |
| **Min-Heap** | A data structure that efficiently maintains the top-K largest values — the smallest element is always on top for quick comparison |
| **WebDataset** | A format for streaming large datasets from sharded `.tar` files, ideal for datasets too large to store locally |
| **Activation Reduction** | Collapsing the spatial dimensions (H×W) of a feature map to a single scalar per channel, using either `max` or `mean` |
| **Checkpointing** | Periodically saving progress to disk so the scan can resume after interruptions |

### Pipeline Diagram

```
ImageNet (1024 shards) → WebDataset Streaming → Preprocessing (Resize/Crop/Normalize)
    → InceptionV1 Forward Pass → Hook captures layer activations
    → Pass 1: Score each channel per image → Maintain Top-K heaps
    → Pass 2: Re-stream to find & save top-K images (full + crop)
```

## Step 1: Install Dependencies

We need three key packages:
- **`webdataset`** — for streaming ImageNet data from HuggingFace Hub without downloading the entire dataset locally
- **`huggingface_hub`** — for authenticating with HuggingFace to access gated datasets
- **`torch-lucent`** — a PyTorch port of Lucid (TensorFlow), providing pre-trained InceptionV1 and visualization tools for interpretability

In [ ]:
# Install webdataset (for streaming ImageNet shards) and huggingface_hub (for authentication)
# The -q flag suppresses verbose output during installation
!pip -q install webdataset huggingface_hub

In [ ]:
# Install torch-lucent: a PyTorch port of the Lucid library (originally TensorFlow).
# Provides pre-trained InceptionV1 with named layers and visualization utilities
# that are essential for mechanistic interpretability research.
!pip install torch-lucent

## Step 2: Import Libraries

We import the following key libraries:
- **`torch`** — PyTorch for model inference and tensor operations
- **`heapq`** — Python's built-in min-heap implementation, used to efficiently maintain the top-K highest scores
- **`webdataset`** — for streaming ImageNet shards without downloading the entire dataset
- **`lucent`** — provides InceptionV1 model and interpretability tools (render, objectives, transforms)
- **`pickle`** — for saving/loading checkpoint data to enable resume capability

In [ ]:
import torch                    # Core deep learning framework
import numpy as np              # Numerical operations (used for image array manipulation)
import heapq                    # Min-heap data structure — efficiently tracks top-K highest scores
from PIL import Image           # Python Imaging Library — for image creation and saving
import matplotlib.pyplot as plt # Plotting library (available for visualization if needed)
import os, re                   # OS utilities (file paths) and regex (filename sanitization)
import webdataset as wds        # WebDataset — streams ImageNet from sharded .tar files on HuggingFace
import pickle as pkl            # Serialization — for saving/loading checkpoints to resume interrupted scans

# Lucent imports — PyTorch port of Google's Lucid interpretability library
from lucent.modelzoo import inceptionv1       # Pre-trained InceptionV1 (GoogLeNet) with named layers
from lucent.optvis import render, param, transform, objectives  # Visualization tools (used in other segments)

## Step 3: Authenticate with HuggingFace Hub

ImageNet-1K is a **gated dataset** on HuggingFace — you need an access token to download/stream it. There are two options provided here:
1. **Interactive login** via `notebook_login()` — opens a widget to paste your token
2. **Manual token** — set `HF_TOKEN` directly as an environment variable

You only need to use **one** of these methods. The token is passed to `curl` commands when streaming the dataset shards.

In [ ]:
# Option A: Interactive login widget — paste your HuggingFace token when prompted.
# This stores the token in your HuggingFace cache so it persists across sessions.
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Option B: Manually set your HuggingFace token as an environment variable.
# Replace the underscore "_" with your actual HuggingFace access token string.
# This env var is read by the curl commands in the WebDataset pipeline below.
HF_TOKEN = "_"
os.environ["HF_TOKEN"] = HF_TOKEN

# Sanity check: print the length of the token to verify it was set (without revealing the token)
print("HF_TOKEN len:", len(os.environ["HF_TOKEN"]))

## Step 4: Set Up ImageNet Data Pipeline

This is the most important data engineering cell. It sets up a **streaming data pipeline** for ImageNet-1K using WebDataset:

### Preprocessing Pipeline
Each image goes through the standard ImageNet preprocessing steps:
1. **Resize** to 256px on the shorter side
2. **Center crop** to 224×224 (the input size InceptionV1 expects)
3. **Convert to tensor** (float values in [0, 1])
4. **Normalize** using ImageNet channel means `[0.485, 0.456, 0.406]` and stds `[0.229, 0.224, 0.225]`

### Streaming Architecture
Instead of downloading all 1.28M images (~150GB), we stream them directly from HuggingFace Hub:
- ImageNet-1K is split into **1024 sharded `.tar` files** (`imagenet1k-train-0000.tar` through `imagenet1k-train-1023.tar`)
- Each shard is fetched on-the-fly using `curl` with authentication
- The `to_tensor` function extracts the image and its **key** (unique identifier) from each sample
- Samples with missing images are filtered out via `.select()`

In [ ]:
import os, webdataset as wds
from torch.utils.data import DataLoader
from torchvision import transforms

# --------------------------------------------------------------------------
# Define the standard ImageNet preprocessing pipeline.
# InceptionV1 expects 224x224 images normalized with these specific statistics.
# --------------------------------------------------------------------------
preprocess = transforms.Compose([
    transforms.Resize(256),                                        # Resize shortest side to 256px
    transforms.CenterCrop(224),                                    # Crop center 224x224 region
    transforms.ToTensor(),                                         # Convert PIL Image → float tensor [0,1]
    transforms.Normalize([0.485, 0.456, 0.406],                   # Subtract ImageNet channel means
                         [0.229, 0.224, 0.225]),                   # Divide by ImageNet channel stds
])

def to_tensor(sample):
    """
    Extract image and key from a WebDataset sample.
    WebDataset stores images under keys like 'jpg', 'jpeg', or 'JPEG'.
    The '__key__' field is a unique identifier for each image in the dataset
    (e.g., 'train/n01440764/n01440764_10026') — we need this to find the
    same image again in Pass 2.
    """
    img = sample.get("jpg") or sample.get("jpeg") or sample.get("JPEG")
    if img is None:
        return None  # Will be filtered out by .select() below
    key = sample.get("__key__", "")  # Unique identifier for this image
    return preprocess(img), key      # Return (preprocessed_tensor, key_string)

# --------------------------------------------------------------------------
# Build the list of URLs for all 1024 ImageNet training shards on HuggingFace.
# Each shard is a .tar file containing ~1,250 images.
# --------------------------------------------------------------------------
base = "https://huggingface.co/datasets/timm/imagenet-1k-wds/resolve/main/"
train_shards = [f"{base}imagenet1k-train-{i:04d}.tar?download=true" for i in range(1024)]

# Wrap each URL in a curl command with authentication header.
# The "pipe:" prefix tells WebDataset to run this shell command and read its stdout.
# --retry 20 and --retry-delay 2 add resilience to network interruptions.
URLS_TRAIN = [
    "pipe:curl -f -sSL --retry 20 --retry-delay 2 "
    "-H \"Authorization: Bearer $HF_TOKEN\" "
    f"\"{u}\""
    for u in train_shards
]

# --------------------------------------------------------------------------
# Create the WebDataset streaming pipeline:
#   1. WebDataset(URLS_TRAIN) — opens each shard URL sequentially
#   2. .decode("pil") — decodes image bytes into PIL Image objects
#   3. .map(to_tensor) — applies our preprocessing and extracts keys
#   4. .select(not None) — drops any samples where the image was missing
# handler=warn_and_continue ensures a single bad sample doesn't crash the run.
# --------------------------------------------------------------------------
dataset_train = (
    wds.WebDataset(URLS_TRAIN, handler=wds.handlers.warn_and_continue)
      .decode("pil", handler=wds.handlers.warn_and_continue)   # Decode images as PIL
      .map(to_tensor, handler=wds.handlers.warn_and_continue)   # Preprocess + extract key
      .select(lambda x: x is not None)                          # Filter out failed samples
)

# Wrap in a standard PyTorch DataLoader.
# batch_size=128: process 128 images at a time through the model.
# num_workers=0: no multiprocessing (streaming handles I/O).
# pin_memory=False: not needed since we're not doing heavy GPU transfers.
dl_train = DataLoader(dataset_train, batch_size=128, num_workers=0, pin_memory=False)

## Step 5: Load the Pre-trained InceptionV1 Model

We load InceptionV1 (GoogLeNet) with pre-trained ImageNet weights from the Lucent library. The model is:
- Moved to GPU if available (falls back to CPU)
- Set to **evaluation mode** (`.eval()`) — this disables dropout and uses running statistics for batch normalization, ensuring deterministic activations

InceptionV1 is the canonical model used in neural network interpretability research (e.g., the original *Feature Visualization* and *Circuits* work by Olah et al.).

In [ ]:
# Select compute device: use GPU if available, otherwise fall back to CPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Load InceptionV1 with pre-trained ImageNet weights from Lucent's model zoo.
# .to(device) moves all parameters to GPU/CPU.
# .eval() sets the model to evaluation mode (disables dropout, uses running BN stats).
model = inceptionv1(pretrained=True).to(device).eval()

## Step 6: Mount Google Drive (Colab Only)

This cell is **only needed in Google Colab**. It mounts your Google Drive at `/content/drive` so that:
- **Checkpoints** can be saved to Drive (surviving Colab session timeouts)
- **Output images** are saved persistently to Drive

If running locally, skip this cell and update the output paths in the configuration below.

In [ ]:
# Mount Google Drive to persist checkpoints and output images across Colab sessions.
# After mounting, your Drive files are accessible at /content/drive/MyDrive/
from google.colab import drive
drive.mount("/content/drive")

## Step 7: Configuration

All key parameters are centralized in a single configuration dictionary. This makes it easy to adjust the experiment without modifying code throughout the notebook.

| Parameter | Description |
|-----------|-------------|
| `LAYER_NAME` | Which InceptionV1 layer to analyze (e.g., `mixed4e` — the 4th Inception module) |
| `CHANNEL_START` / `CHANNEL_END` | Range of channels to scan. `mixed4e` has 832 channels (0–831) |
| `TOPK` | How many top-activating images to keep per channel (10 = keep the 10 strongest) |
| `HEAP_BATCH` | How many candidates from each batch to consider for the heap (20 = compare top-20 per batch) |
| `REDUCTION` | How to collapse spatial dimensions: `"max"` (strongest activation anywhere) or `"mean"` (average across all locations) |
| `OUT_ROOT` | Directory where output images will be saved |
| `CROP_FRAC` | Fraction of the image to crop around the max-activation point (0.25 = 25% of image dimensions) |

In [ ]:
# ============================================================================
# Central configuration dictionary — all experiment parameters in one place.
# Modify LAYER_NAME and CHANNEL_START/CHANNEL_END based on your assigned range.
# ============================================================================
CFG = {
    # Which layer to extract activations from.
    # InceptionV1 layers include: mixed3a, mixed3b, mixed4a, mixed4b, mixed4c, mixed4d, mixed4e, mixed5a, mixed5b
    "LAYER_NAME": "mixed4e",

    # Channel range to analyze (inclusive). mixed4e has 832 channels (indices 0–831).
    # For distributed work, different contributors can scan different channel ranges.
    "CHANNEL_START": 0,
    "CHANNEL_END": 832,

    # Number of top-activating images to keep per channel.
    # 10 means we find the 10 images (out of ~1.28M) that activate each channel the most.
    "TOPK": 10,

    # Number of top candidates to extract from each batch for heap comparison.
    # Must be >= TOPK. Higher values are slightly slower but may catch more candidates.
    "HEAP_BATCH": 20,

    # How to reduce spatial dimensions (H×W) to a single score per channel:
    # "max"  — use the strongest activation at any spatial location (good for localized features)
    # "mean" — average across all spatial locations (good for distributed/texture features)
    "REDUCTION": "max",

    # Root directory for saving output images (adjust for your setup).
    "OUT_ROOT": "/content/drive/MyDrive/Dataset_images",

    # Fraction of image dimensions for the crop window around the max-activation point.
    # 0.25 means the crop is 25% of the image height × 25% of the image width.
    "CROP_FRAC": 0.25,
}

## Step 8: Image Helper Functions

These utility functions handle the reverse transformation from normalized tensors back to displayable/saveable images:

### `tensor_to_pil(x)`
Converts a normalized image tensor back to a PIL Image by:
1. **Denormalizing** — reversing the ImageNet mean/std normalization
2. **Clamping** — ensuring pixel values stay in [0, 1]
3. **Converting** — from float tensor to uint8 numpy array to PIL Image

### `crop_from_tensor_and_feat(img_tensor_norm, feat_single, channel_id, frac)`
Creates a **spatially-targeted crop** of the image, centered on where the channel's activation is strongest:
1. Finds the **(iy, ix)** location of maximum activation in the feature map
2. Maps that location back to **pixel coordinates** in the original image (accounting for the spatial downsampling in the network)
3. Crops a window of size `frac × image_dimensions` centered on that point

This crop highlights **exactly what part of the image** the channel is responding to.

In [ ]:
# ImageNet normalization constants — needed to reverse the preprocessing and recover original pixel values.
# Shape [3,1,1] allows broadcasting over spatial dimensions (H, W).
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)  # Per-channel means
std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)  # Per-channel standard deviations


def tensor_to_pil(x: torch.Tensor) -> Image.Image:
    """
    Convert a normalized image tensor back to a PIL Image.

    Args:
        x: Tensor of shape [C,H,W] or [1,C,H,W] — a single normalized image.

    Returns:
        PIL Image with uint8 pixel values in [0, 255].
    """
    x = x.detach().cpu()       # Move to CPU and detach from computation graph

    if x.dim() == 4:
        x = x[0]              # Remove batch dimension: [1,C,H,W] → [C,H,W]

    # Reverse normalization: pixel = (normalized * std) + mean
    x = (x * std + mean).clamp(0, 1)

    # Convert from [C,H,W] float tensor → [H,W,C] uint8 numpy array → PIL Image
    x = (x.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    return Image.fromarray(x)


def crop_from_tensor_and_feat(img_tensor_norm: torch.Tensor,
                              feat_single: torch.Tensor,
                              channel_id: int,
                              frac: float = 0.25) -> Image.Image:
    """
    Crop a region of the image centered on the spatial location of maximum activation
    for a given channel. This shows WHAT part of the image the channel is responding to.

    Args:
        img_tensor_norm: Normalized image tensor [C,H,W] (single image, not batched).
        feat_single:     Feature map tensor [C,h,w] for this image at the target layer.
        channel_id:      Which channel to find the max activation for.
        frac:            Fraction of image size for the crop window (0.25 = 25%).

    Returns:
        PIL Image of the cropped region.
    """
    # Step 1: Denormalize the image back to [0, 255] uint8 numpy array
    img = (img_tensor_norm.detach().cpu() * std + mean).clamp(0, 1)
    img_np = (img.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    H, W = img_np.shape[:2]  # Original image dimensions (224, 224)

    # Step 2: Extract the activation map for this specific channel
    # Clamp to zero to ignore negative activations (ReLU-like behavior)
    act = feat_single[channel_id].detach().cpu().clamp(min=0)  # Shape: [h, w] (e.g., [14, 14])
    h, w = act.shape  # Feature map spatial dimensions (smaller than image due to pooling/strides)

    # Step 3: Find the (row, col) of the maximum activation in the feature map
    flat = int(torch.argmax(act).item())  # Flatten index of the max value
    iy, ix = flat // w, flat % w          # Convert flat index to 2D coordinates

    # Step 4: Map feature-map coordinates back to pixel coordinates in the original image.
    # The +0.5 centers the point within the feature map cell.
    # Multiplying by H/h (or W/w) scales from feature map space to pixel space.
    cy = int((iy + 0.5) * H / h)  # Center y-coordinate in pixel space
    cx = int((ix + 0.5) * W / w)  # Center x-coordinate in pixel space

    # Step 5: Compute crop boundaries, clamped to image edges
    ch, cw = int(H * frac), int(W * frac)                   # Crop height and width in pixels
    y0 = max(0, cy - ch // 2); y1 = min(H, y0 + ch)         # Top and bottom boundaries
    x0 = max(0, cx - cw // 2); x1 = min(W, x0 + cw)         # Left and right boundaries

    # Step 6: Extract and return the cropped region
    return Image.fromarray(img_np[y0:y1, x0:x1])

## Step 9: Core Scoring Helper — Activation Reduction

This function collapses a 4D feature map `[B, C, H, W]` into a 2D score matrix `[B, C]` by reducing the spatial dimensions.

**Why do we need this?** Each channel produces a 2D activation map (e.g., 14×14 for `mixed4e`). To rank how strongly an image activates a channel, we need a single scalar score per (image, channel) pair. Two strategies:

- **`max`** — Takes the maximum activation across all spatial positions. Best for **localized features** (e.g., eyes, wheels) that might only appear in one part of the image.
- **`mean`** — Averages all spatial positions. Better for **distributed features** (e.g., textures, colors) that span the whole image.

In [ ]:
@torch.no_grad()  # Disable gradient computation — we're only doing inference, saves memory
def reduce_all_channels(feat: torch.Tensor, reduction: str) -> torch.Tensor:
    """
    Reduce a feature map from [B, C, H, W] → [B, C] by collapsing spatial dimensions.

    Args:
        feat:      Feature map tensor of shape [B, C, H, W] from a convolutional layer.
        reduction: Strategy for spatial reduction — "max" or "mean".

    Returns:
        Tensor of shape [B, C] where each entry is the score for (image_i, channel_j).
    """
    if reduction == "max":
        # amax over dims (2,3) = take the maximum value across all H×W spatial positions.
        # This captures the STRONGEST activation for each channel, regardless of where it occurs.
        return feat.amax(dim=(2, 3))
    elif reduction == "mean":
        # Average across all H×W spatial positions.
        # This gives a holistic measure of how much the channel fires across the whole image.
        return feat.mean(dim=(2, 3))
    else:
        raise ValueError("reduction must be 'max' or 'mean'")

## Step 10: Pass 1 — Scan the Entire Dataset to Find Top-K Image Keys

This is the **core algorithm** of the notebook. It streams through all ~1.28 million ImageNet images in a single pass and, for each channel, identifies the top-K images with the highest activation scores.

### Algorithm

For each batch of images:
1. Run a **forward pass** through the model
2. A **forward hook** intercepts the activations at the target layer
3. **Reduce** each channel's spatial activation map to a single score (via max or mean)
4. For each channel, compare the batch's top scores against a **min-heap** of size K
5. If a score is higher than the current minimum in the heap, replace it

### Why a Min-Heap?

A min-heap of size K keeps the K largest values seen so far, with the **smallest of the K** always on top. This allows O(log K) comparisons per candidate — far more efficient than sorting all 1.28M scores.

### Checkpointing

Since scanning ~10,000 batches takes hours (even on a GPU), the function supports **checkpointing**:
- Every `SAVE_EVERY` batches, the current heap state is pickled to disk
- On restart, it loads the checkpoint and skips already-processed batches
- This is critical for long Colab sessions that may time out

### Key Outputs

The function returns `topk_by_channel`: a dictionary mapping each channel index to a sorted list of `(score, image_key)` tuples — the top-K highest-activating images for that channel.

In [ ]:
# ============================================================================
# Pass 1: Scan the entire ImageNet training set ONCE to find the top-K
# highest-activating image keys for each channel in the target layer.
# ============================================================================

# Checkpoint path — save progress here so we can resume after interruptions
CKPT_PATH = "/content/drive/MyDrive/topk_scan_ckpt.pkl"
SAVE_EVERY = 1000  # Save a checkpoint every 1000 batches


def save_ckpt(path, step, heaps, ch0, ch1):
    """Save current scan progress to a pickle file."""
    with open(path, "wb") as f:
        pkl.dump({"step": step, "heaps": heaps, "ch0": ch0, "ch1": ch1}, f)


def load_ckpt(path):
    """Load a previously saved checkpoint, or return None if no checkpoint exists."""
    if os.path.exists(path):
        with open(path, "rb") as f:
            return pkl.load(f)
    return None


def scan_topk_keys(model, dl_train, device, layer_name: str,
                   channel_start: int, channel_end_inclusive: int,
                   topk: int, heap_batch: int, reduction: str):
    """
    Stream through the entire dataset and find the top-K highest-activating
    image keys for each channel in the specified layer.

    Args:
        model:                The InceptionV1 model (already on device, in eval mode).
        dl_train:             DataLoader that streams ImageNet training images.
        device:               torch.device (cuda or cpu).
        layer_name:           Name of the layer to hook (e.g., "mixed4e").
        channel_start:        First channel index to analyze (inclusive).
        channel_end_inclusive: Last channel index to analyze (inclusive).
        topk:                 Number of top-activating images to keep per channel.
        heap_batch:           How many top candidates to pull from each batch.
        reduction:            "max" or "mean" — how to reduce spatial dimensions.

    Returns:
        topk_by_channel: dict mapping channel_id → list of (score, key) tuples,
                         sorted by score in descending order.
    """
    # ---- Register a forward hook to capture intermediate activations ----
    # A "forward hook" is a callback that PyTorch calls AFTER a layer computes
    # its output during a forward pass. We use it to grab the activation tensor
    # without modifying the model code.
    activation = {}  # Mutable container to store the hooked activation

    def hook_fn(module, inp, out):
        """Hook callback: store the layer's output tensor."""
        activation["feat"] = out

    # Look up the target layer by name and attach the hook
    layer = dict(model.named_modules())[layer_name]
    handle = layer.register_forward_hook(hook_fn)

    model.eval()  # Ensure evaluation mode

    # ---- Initialize or resume from checkpoint ----
    heaps = None    # Dict of min-heaps, one per channel: {channel_id: [(score, key), ...]}
    ch0 = None      # Actual start channel (after clamping to layer's channel count)
    ch1 = None      # Actual end channel (after clamping)
    start_step = 0  # Which batch to start from (0 = beginning)

    # Try to load a previous checkpoint to resume an interrupted scan
    ckpt = load_ckpt(CKPT_PATH)
    if ckpt is not None:
        start_step = int(ckpt["step"]) + 1  # Resume from the NEXT batch
        heaps = ckpt["heaps"]
        ch0 = int(ckpt["ch0"])
        ch1 = int(ckpt["ch1"])
        print(f"Resuming from batch {start_step} (range [{ch0}, {ch1}])")

    last_step = None  # Track the last processed batch for final checkpoint

    # ---- Main scan loop ----
    with torch.no_grad():  # No gradients needed — pure inference
        for step, (imgs, keys) in enumerate(dl_train):

            # Skip batches that were already processed in a previous run
            if step < start_step:
                continue

            last_step = step

            # Move batch of images to GPU/CPU
            imgs = imgs.to(device, non_blocking=True)

            # Run forward pass through the model.
            # The hook_fn will automatically capture the activations at our target layer.
            use_cuda = imgs.is_cuda
            with torch.amp.autocast(device_type="cuda", enabled=use_cuda):
                _ = model(imgs)  # We don't need the model's final output

            # Retrieve the hooked activation: shape [B, C, h, w]
            # B = batch size, C = number of channels, h×w = spatial dimensions
            feat = activation["feat"]
            B, C = feat.shape[0], feat.shape[1]

            # ---- First-batch initialization: set up heaps and channel range ----
            if heaps is None:
                # Clamp the requested channel range to the layer's actual channel count
                ch0 = max(0, int(channel_start))
                ch1 = min(C - 1, int(channel_end_inclusive))
                if ch0 > ch1:
                    handle.remove()
                    raise ValueError(f"Invalid channel range after clamping: [{ch0}, {ch1}] with C={C}")

                # Create an empty min-heap for each channel in our range
                heaps = {c: [] for c in range(ch0, ch1 + 1)}
                print(f"Scanning layer={layer_name} channels={C} using range [{ch0}, {ch1}]")

            else:
                # Sanity check: if resuming, ensure checkpoint's channel range is valid
                ch0 = max(0, min(ch0, C - 1))
                ch1 = max(0, min(ch1, C - 1))
                if ch0 > ch1:
                    handle.remove()
                    raise ValueError(f"Checkpoint channel range invalid for this run: [{ch0}, {ch1}] with C={C}")

            # ---- Compute per-channel scores for all images in this batch ----
            # Reduce [B, C, h, w] → [B, C] using max or mean over spatial dims
            scores_bc = reduce_all_channels(feat, reduction).float()  # [B, C]

            # Extract only the channels we care about: [B, num_channels_in_range]
            scores_sel = scores_bc[:, ch0:ch1 + 1]

            # Get the top-k candidates FROM THIS BATCH (not global top-k yet)
            # k = min(heap_batch, B) handles the case where batch < heap_batch
            k = min(heap_batch, B)
            vals, idxs = torch.topk(scores_sel, k=k, dim=0)  # [k, num_channels]

            vals = vals.detach().cpu()
            idxs = idxs.detach().cpu()

            # ---- Update the min-heaps with this batch's candidates ----
            for offset, c in enumerate(range(ch0, ch1 + 1)):
                h = heaps[c]  # The min-heap for this channel
                for j in range(k):
                    v = float(vals[j, offset].item())  # Activation score
                    i = int(idxs[j, offset].item())    # Index within the batch
                    key = keys[i]                       # Image key (unique ID)

                    item = (v, key)
                    if len(h) < topk:
                        # Heap not full yet — just add it
                        heapq.heappush(h, item)
                    else:
                        # Heap is full — only add if this score beats the current minimum.
                        # heapreplace pops the min and pushes the new item in O(log K).
                        if v > h[0][0]:
                            heapq.heapreplace(h, item)

            # ---- Progress logging every 50 batches ----
            if (step + 1) % 50 == 0:
                sample_c = ch0
                if heaps[sample_c]:
                    print(f"Scanning batches={step+1} ch{sample_c} min_top{topk}={heaps[sample_c][0][0]:.4f}")

            # ---- Periodic checkpoint save ----
            if (step + 1) % SAVE_EVERY == 0:
                save_ckpt(CKPT_PATH, step, heaps, ch0, ch1)
                print(f"ckpt saved at batch {step}")

    # ---- Cleanup: remove the forward hook ----
    handle.remove()

    # ---- Final checkpoint save ----
    if heaps is not None and last_step is not None:
        save_ckpt(CKPT_PATH, last_step, heaps, ch0, ch1)
        print(f"ckpt final save at batch {last_step}")

    # ---- Convert min-heaps to sorted lists (descending by score) ----
    # heapq gives us a min-heap, so we sort descending to get rank-1 = highest score
    topk_by_channel = {
        c: sorted(h, key=lambda x: x[0], reverse=True)
        for c, h in heaps.items()
    }
    return topk_by_channel

## Step 11: Pass 2 — Retrieve and Save Top-K Images

After Pass 1 identified *which* images to save (by their keys), Pass 2 streams through the dataset again to **find those specific images** and save them to disk.

### Why a second pass?

During Pass 1, we only stored image **keys** (identifiers), not the actual image data — storing 832 channels × 10 images × full tensors in memory would be prohibitive. Instead, we do a second scan, looking only for the specific images we identified.

### What gets saved?

For each top-K image, two files are saved per channel:
1. **Full image** — `rank01_FULL_score3.1415_imagename.jpg` — the complete 224×224 image
2. **Crop image** — `rank01_CROP_score3.1415_imagename.jpg` — a zoomed-in crop centered on the location of maximum activation

### Output directory structure
```
OUT_ROOT/
  mixed4e/
    ch_0000/
      rank01_FULL_score5.2341_n01440764_10026.jpg
      rank01_CROP_score5.2341_n01440764_10026.jpg
      rank02_FULL_score4.8912_n01440764_2345.jpg
      ...
    ch_0001/
      ...
```

### Early exit optimization
Once all wanted images have been found, the scan stops immediately — no need to process remaining shards.

In [ ]:
# ============================================================================
# Pass 2: Stream through the dataset again, find the images identified in
# Pass 1, and save full + cropped versions to disk.
# ============================================================================


def map_top_keys(topk_by_channel):
    """
    Build a reverse index from image keys to their target channels.

    From topk_by_channel (channel → list of (score, key)), we create:
      - wanted_keys: set of all unique image keys we need to find
      - key_to_targets: dict mapping each key → list of (channel, score, rank)
        (one image may be top-K for multiple channels)

    Returns:
        wanted_keys:    set of image key strings
        key_to_targets: dict mapping key → [(channel_id, score, rank), ...]
    """
    wanted_keys = set()
    key_to_targets = {}  # key → list[(channel, score, rank)]

    for c, items in topk_by_channel.items():
        # items are sorted descending by score from Pass 1
        for rank, (score, key) in enumerate(items, start=1):
            wanted_keys.add(key)
            key_to_targets.setdefault(key, []).append((c, float(score), rank))

    return wanted_keys, key_to_targets


def _safe_filename(s: str, max_len: int = 120) -> str:
    """
    Sanitize a string for use as a filename.
    Replaces slashes and special characters with underscores,
    and truncates to max_len characters.
    """
    s = str(s).replace("/", "_").replace("\\", "_")
    s = re.sub(r"[^a-zA-Z0-9._-]+", "_", s)
    return s[:max_len]


@torch.no_grad()  # No gradients needed — pure inference
def save_topk_images_wds_streaming(
    model, dl_train, device,
    layer_name: str,
    topk_by_channel: dict,
    out_root: str,
    crop_frac: float,
):
    """
    Stream through the dataset, find images matching the top-K keys from Pass 1,
    and save both full images and activation-centered crops to disk.

    Args:
        model:             InceptionV1 model (on device, eval mode).
        dl_train:          DataLoader streaming ImageNet training images.
        device:            torch.device.
        layer_name:        Layer to hook for spatial activation (same as Pass 1).
        topk_by_channel:   Output from scan_topk_keys — channel → [(score, key), ...].
        out_root:          Root directory for saving images.
        crop_frac:         Fraction of image to crop around max activation point.
    """
    # Build the reverse index: which keys to look for, and what channels need them
    wanted_keys, key_to_targets = map_top_keys(topk_by_channel)
    remaining = set(wanted_keys)  # Track which keys we still haven't found

    # ---- Register forward hook (same as Pass 1) ----
    # We need activations again to compute crops centered on max-activation locations
    activation = {}

    def hook_fn(module, inp, out):
        activation["feat"] = out

    layer = dict(model.named_modules())[layer_name]
    handle = layer.register_forward_hook(hook_fn)

    # Create the output directory for this layer
    layer_dir = os.path.join(out_root, layer_name)
    os.makedirs(layer_dir, exist_ok=True)

    model.eval()

    # ---- Stream through the dataset ----
    for step, (imgs, keys) in enumerate(dl_train):
        # Forward pass to get activations (needed for crop computation)
        imgs = imgs.to(device, non_blocking=True)

        use_cuda = imgs.is_cuda
        with torch.amp.autocast(device_type="cuda", enabled=use_cuda):
            _ = model(imgs)

        feat = activation["feat"]  # [B, C, h, w]

        # ---- Check which images in this batch are ones we're looking for ----
        hit_indices = [i for i, k in enumerate(keys) if k in remaining]
        if not hit_indices:
            # No wanted images in this batch — skip it
            if (step + 1) % 200 == 0:
                print(f"batches={step+1} remaining={len(remaining)}")
            continue

        # ---- Process each found image ----
        for i in hit_indices:
            key = keys[i]
            key_safe = _safe_filename(key)  # Sanitize for filesystem

            # Convert the full image tensor back to a PIL Image (denormalized)
            pil_full = tensor_to_pil(imgs[i])

            # Save for ALL channels that identified this image as top-K.
            # (One image might be top-K for multiple channels.)
            for (c, score, rank) in key_to_targets[key]:
                # Create per-channel subdirectory: e.g., mixed4e/ch_0042/
                ch_dir = os.path.join(layer_dir, f"ch_{c:04d}")
                os.makedirs(ch_dir, exist_ok=True)

                # Compute the activation-centered crop for THIS specific channel
                pil_crop = crop_from_tensor_and_feat(imgs[i], feat[i], c, frac=crop_frac)

                # Save full image: e.g., rank01_FULL_score5.2341_n01440764_10026.jpg
                pil_full.save(os.path.join(ch_dir, f"rank{rank:02d}_FULL_score{score:.4f}_{key_safe}.jpg"))

                # Save cropped image: e.g., rank01_CROP_score5.2341_n01440764_10026.jpg
                pil_crop.save(os.path.join(ch_dir, f"rank{rank:02d}_CROP_score{score:.4f}_{key_safe}.jpg"))

            # Mark this key as found — don't look for it again
            remaining.remove(key)

        # ---- Progress logging ----
        if (step + 1) % 50 == 0:
            print(f"batches={step+1} saved={(len(wanted_keys) - len(remaining))}/{len(wanted_keys)} remaining={len(remaining)}")

        # ---- Early exit: stop once ALL wanted images have been found ----
        if not remaining:
            print(f"Done early at batch {step+1}")
            break

    # Cleanup: remove the forward hook
    handle.remove()
    print("Done ->", layer_dir)

## Step 12: Execute the Full Pipeline

This cell runs both passes end-to-end:

1. **`scan_topk_keys()`** — Pass 1: Streams through all ~1.28M images, scores every channel, and identifies the top-10 image keys per channel using min-heaps. Saves checkpoints every 1000 batches.

2. **`save_topk_images_wds_streaming()`** — Pass 2: Streams through the dataset again, finds the specific images identified in Pass 1, and saves full + cropped versions organized by channel.

**Expected runtime:** Several hours on a GPU (processing ~10,000 batches of 128 images each, twice). Progress is printed every 50 batches. If the session is interrupted, re-running this cell will resume from the last checkpoint.

In [ ]:
# ============================================================================
# PASS 1: Scan the entire ImageNet training set to find the top-K image keys
# per channel. This only stores keys (strings), not actual image data.
# ============================================================================
topk_by_channel = scan_topk_keys(
    model=model,
    dl_train=dl_train,
    device=device,
    layer_name=CFG["LAYER_NAME"],           # e.g., "mixed4e"
    channel_start=CFG["CHANNEL_START"],      # e.g., 0
    channel_end_inclusive=CFG["CHANNEL_END"], # e.g., 832
    topk=CFG["TOPK"],                        # e.g., 10 images per channel
    heap_batch=CFG["HEAP_BATCH"],            # e.g., 20 candidates per batch
    reduction=CFG["REDUCTION"],              # e.g., "max"
)

# ============================================================================
# PASS 2: Stream through the dataset again, find the images by key, and save
# both the full images and activation-centered crops to disk/Drive.
# ============================================================================
save_topk_images_wds_streaming(
    model=model,
    dl_train=dl_train,
    device=device,
    layer_name=CFG["LAYER_NAME"],            # Same layer as Pass 1
    topk_by_channel=topk_by_channel,         # The top-K results from Pass 1
    out_root=CFG["OUT_ROOT"],                # Where to save images
    crop_frac=CFG["CROP_FRAC"],              # Crop window size (fraction of image)
)